### 1- Setup & Installations

In [2]:
%pip install pandas duckdb huggingface_hub pyarrow

  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 6.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 10.0 MB/s eta 0:00:0000:0100:01
Using cached click-8.4.1-py3-none-any.whl (116 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 9.4 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Extraction of the data through Parquet method or live API


#### i. Parquet Extraction

In [2]:
#Run this (mac_user: python3 extract_parquet.py) in the terminal to extract the data from the OFF API, transform it, 
# and save as CSV (and optionally SQLite):

 

In [4]:
import pandas as pd
from pathlib import Path

IN_PATH = Path("..") / "data" / "products_europe_full.csv"  
df_par = pd.read_csv(IN_PATH, dtype={"code": str})

print(f"{len(df_par):,} rows loaded <- {IN_PATH.resolve()}")
print(f"Columns: {list(df_par.columns)}")
print("\nRows per country:")
print(df_par["country"].value_counts().to_string())
df_par.head()

2,620,517 rows loaded <- /Users/kanakyadav/Documents/GitHub/capstone/data/products_europe_full.csv
Columns: ['code', 'product_name', 'brands', 'country', 'country_tag', 'nutriscore_grade', 'nutriscore_score', 'nova_group', 'eco_grade', 'additives_n', 'energy_kcal_100g', 'sugars_100g', 'fat_100g', 'saturated_fat_100g', 'salt_100g', 'sodium_100g', 'proteins_100g', 'fiber_100g', 'completeness', 'last_modified']

Rows per country:
country
France            1119309
Germany            401528
Spain              337480
Italy              261832
United Kingdom     172115
Switzerland        100606
Belgium             94962
Netherlands         76159
Poland              32195
Sweden              24331


,code,product_name,brands,country,country_tag,nutriscore_grade,nutriscore_score,nova_group,eco_grade,additives_n,energy_kcal_100g,sugars_100g,fat_100g,saturated_fat_100g,salt_100g,sodium_100g,proteins_100g,fiber_100g,completeness,last_modified
0,0000101209159,Véritable pâte à tartiner noisettes chocolat noir,Bovetti,France,en:france,e,25.0,NaN,d,NaN,617.0,32.0,48.0,10.00,0.01,0.004,8.00,NaN,0.6875,2026-03-25 19:05:13+01:00
1,0000130008136,Escalope de dinde,NaN,France,en:france,unknown,NaN,NaN,e,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3000,2019-01-04 21:45:08+01:00
2,0000140323687,Madeleine Framboise,NaN,France,en:france,unknown,NaN,NaN,b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3750,2023-04-29 01:59:07+02:00
3,0000141013129,Croissants margarine,NaN,France,en:france,unknown,NaN,NaN,b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2750,2025-05-31 05:53:16+02:00
4,0000171812457,Glaces vegetales des alpes,NaN,France,en:france,unknown,NaN,NaN,unknown,NaN,200.0,26.6,9.2,6.21,0.10,0.040,2.22,NaN,0.2750,2026-03-25 19:05:13+01:00


#### ii. Through Live API 

In [8]:
import time
import requests
import pandas as pd
from pathlib import Path


USER_AGENT = "Capstone/1.0 (kanakyadav235@gmail.com)"

SEARCH_URL = "https://world.openfoodfacts.org/api/v2/search"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
DATA_DIR.mkdir(exist_ok=True)
OUT_PATH = DATA_DIR / "products_europe_api.csv"

In [4]:
COUNTRIES = {   # display name : value OFF expects in countries_tags_en
    "Germany": "Germany", "United Kingdom": "United Kingdom", "France": "France",
    "Italy": "Italy", "Spain": "Spain", "Netherlands": "Netherlands",
    "Switzerland": "Switzerland", "Poland": "Poland", "Belgium": "Belgium",
    "Sweden": "Sweden",
}

FIELDS = ",".join([
    "code", "product_name", "brands", "countries_tags",
    "nutriscore_grade", "nutriscore_score", "nova_group",
    "environmental_score_grade", "additives_n", "completeness",
    "last_modified_t", "nutriments",
])

PAGE_SIZE = 100      # products per page (OFF's reliable max)
MAX_PAGES = 5        # 5 x 100 = up to 500 products PER country (a sample!)
SLEEP_SECONDS = 6.5  # stay under the 10-requests/minute search limit

In [5]:
def fetch_country(country_value: str) -> list[dict]:
    rows = []
    for page in range(1, MAX_PAGES + 1):
        params = {"countries_tags_en": country_value, "fields": FIELDS,
                  "page_size": PAGE_SIZE, "page": page}
        try:
            r = requests.get(SEARCH_URL, params=params,
                             headers={"User-Agent": USER_AGENT}, timeout=30)
            if r.status_code in (429, 503):
                print(f"    rate-limited (HTTP {r.status_code}); waiting 30s...")
                time.sleep(30)
                continue
            r.raise_for_status()
            products = r.json().get("products", [])
        except requests.RequestException as e:
            print(f"    request failed on page {page}: {e}")
            break
        if not products:
            break
        rows.extend(products)
        print(f"    page {page}: {len(products)} products (running total {len(rows)})")
        time.sleep(SLEEP_SECONDS)   # be polite
    return rows


def flatten(prod: dict, country_name: str) -> dict:
    n = prod.get("nutriments", {}) or {}
    name = prod.get("product_name")
    if isinstance(name, list):              # some responses give a list of langs
        name = name[0].get("text") if name else None
    return {
        "code": prod.get("code"),
        "product_name": name,
        "brands": prod.get("brands"),
        "country": country_name,
        "nutriscore_grade": prod.get("nutriscore_grade"),
        "nutriscore_score": prod.get("nutriscore_score"),
        "nova_group": prod.get("nova_group"),
        "eco_grade": prod.get("environmental_score_grade"),
        "additives_n": prod.get("additives_n"),
        "energy_kcal_100g": n.get("energy-kcal_100g"),
        "sugars_100g": n.get("sugars_100g"),
        "fat_100g": n.get("fat_100g"),
        "saturated_fat_100g": n.get("saturated-fat_100g"),
        "salt_100g": n.get("salt_100g"),
        "sodium_100g": n.get("sodium_100g"),
        "proteins_100g": n.get("proteins_100g"),
        "fiber_100g": n.get("fiber_100g"),
        "completeness": prod.get("completeness"),
        "last_modified_t": prod.get("last_modified_t"),
    }

In [6]:
all_rows = []
for display, value in COUNTRIES.items():
    print(f"Fetching {display}...")
    for prod in fetch_country(value):
        all_rows.append(flatten(prod, display))

print(f"\nCollected {len(all_rows)} raw rows.")

Fetching Germany...
    rate-limited (HTTP 503); waiting 30s...
    rate-limited (HTTP 503); waiting 30s...
    page 3: 100 products (running total 100)
    page 4: 100 products (running total 200)
    rate-limited (HTTP 503); waiting 30s...
Fetching United Kingdom...
    page 1: 100 products (running total 100)
    page 2: 100 products (running total 200)
    page 3: 100 products (running total 300)
    rate-limited (HTTP 503); waiting 30s...
    rate-limited (HTTP 503); waiting 30s...
Fetching France...
    rate-limited (HTTP 503); waiting 30s...
    page 2: 100 products (running total 100)
    page 3: 100 products (running total 200)
    rate-limited (HTTP 503); waiting 30s...
    page 5: 100 products (running total 300)
Fetching Italy...
    rate-limited (HTTP 503); waiting 30s...
    rate-limited (HTTP 503); waiting 30s...
    page 3: 100 products (running total 100)
    rate-limited (HTTP 503); waiting 30s...
    page 5: 100 products (running total 200)
Fetching Spain...
    page

In [9]:
df_api = pd.read_csv("../data/products_europe_api.csv")

# drop nameless products and exact (code, country) duplicates
df_api = df_api[df_api["product_name"].notna() & (df_api["product_name"].astype(str).str.strip() != "")]
df_api = df_api.drop_duplicates(subset=["code", "country"]).reset_index(drop=True)

df_api.to_csv(OUT_PATH, index=False)
print(f"{len(df_api):,} rows written -> {OUT_PATH.resolve()}")
print("\nRows per country:")
print(df_api["country"].value_counts().to_string())

3,031 rows written -> /Users/kanakyadav/Documents/GitHub/capstone/data/products_europe_api.csv

Rows per country:
country
Sweden            394
Belgium           389
Spain             388
United Kingdom    300
France            299
Netherlands       294
Switzerland       291
Poland            278
Germany           200
Italy             198


### 3. Data cleaning & Wrangling

| **Next Notebook -> 02_data_cleaning&wrangling**

- I've decide to go with the **products_europe.csv** file extrated through parquet method (check: extract_parquet.py). It has 2,620,517 rows. A good data for prediction and forecast models. 

- Approx. 2 million products are too much, need to narrow it down. I'm taking 7k products from each countries, so will make it 70k products from top 10 european countries (Top is selected as per GDP)